# Thí nghiệm Hybrid 1D-CNN + LSTM

Notebook này chạy lại pipeline trong `CNN_LSTM.py`. Kết quả được lưu vào `artifacts_cnn_lstm_rerun/` để không ghi đè model gốc trong `artifacts/`.

In [3]:
from pathlib import Path

print('Working directory:', Path.cwd())
required = [
    'CNN_LSTM.py',
    '../SQLInjection_XSS_MixDataset.1.0.0.csv',
    '../csic_database.csv',
    '../obfuscation_dataset_full.xlsx',
]
for path in required:
    assert Path(path).exists(), f'Thiếu file: {path}'
print('Đã tìm thấy đầy đủ code và dataset.')

Working directory: c:\Users\Admin\OneDrive\Desktop\NCKH_sp
Đã tìm thấy đầy đủ code và dataset.


## 1. Kiểm tra kiến trúc
Cell này chỉ dựng model và in shape, không huấn luyện.

In [4]:
import importlib.util
import sys
from pathlib import Path

module_path = Path('CNN_LSTM.py').resolve()
spec = importlib.util.spec_from_file_location('nckh_test_pipeline', module_path)
pipeline = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = pipeline
spec.loader.exec_module(pipeline)

demo_model = pipeline.build_model(vocab_size=191, max_len=1024, embedding_dim=64)
demo_model.summary()

Model: "Hybrid_1D_CNN_LSTM_Web_Attack_Detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_embedding (Embedding)      │ (None, 1024, 64)       │        12,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3 (Conv1D)                │ (None, 1024, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5 (Conv1D)                │ (None, 256, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling1D)           │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_context (LSTM)             │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_classifier (Dense)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attack_probability (Dense)      │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 258,881 (1011.25 KB)

 Trainable params: 258,881 (1011.25 KB)

 Non-trainable params: 0 (0.00 B)

## 2. Smoke test tùy chọn
Chạy trên mẫu nhỏ để kiểm tra môi trường. Không dùng kết quả này trong báo cáo.

In [5]:
%run CNN_LSTM.py --sample-size 3000 --obfu-sample-size 1000 --epochs 3 --batch-size 128 --output-dir artifacts_cnn_lstm_smoke

=== DATA PREPARED ===
Train           : 2,160
Val             : 240
Test            : 600
Obfuscated test : 1,000
Base p99 length : 974

=== TOKENIZER ===
Vocabulary size: 111
Tokenizer was fit on train payloads only.

=== INPUT SHAPES ===
X_train: (2160, 1024)
X_val  : (240, 1024)
X_test : (600, 1024)
X_obfu : (1000, 1024)
Class weights: {0: 1.4210526315789473, 1: 0.7714285714285715}


Model: "Hybrid_1D_CNN_LSTM_Web_Attack_Detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_embedding (Embedding)      │ (None, 1024, 64)       │         7,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3 (Conv1D)                │ (None, 1024, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5 (Conv1D)                │ (None, 256, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling1D)           │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_context (LSTM)             │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_classifier (Dense)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attack_probability (Dense)      │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 253,761 (991.25 KB)

 Trainable params: 253,761 (991.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 330ms/step - accuracy: 0.5219 - auc: 0.5667 - loss: 0.6823 - precision: 0.6898 - recall: 0.4752
Epoch 1: val_loss improved from None to 0.53580, saving model to artifacts_cnn_lstm_smoke\best_hybrid_cnn_lstm.keras

Epoch 1: finished saving model to artifacts_cnn_lstm_smoke\best_hybrid_cnn_lstm.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 383ms/step - accuracy: 0.5968 - auc: 0.6855 - loss: 0.6451 - precision: 0.7702 - recall: 0.5386 - val_accuracy: 0.8000 - val_auc: 0.8788 - val_loss: 0.5358 - val_precision: 0.8857 - val_recall: 0.7949
Epoch 2/3
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 325ms/step - accuracy: 0.8166 - auc: 0.7978 - loss: 0.5099 - precision: 0.8571 - recall: 0.8584
Epoch 2: val_loss improved from 0.53580 to 0.47582, saving model to artifacts_cnn_lstm_smoke\best_hybrid_cnn_lstm.keras

Epoch 2: finished saving model to artifacts_cnn_lstm_smoke\best_hybrid_cnn_lstm.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 6s 346ms/step - accuracy: 0.8088 - auc: 0.8090 - loss: 0.50

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


## 3. Huấn luyện đầy đủ
Cell này chạy toàn bộ dữ liệu. EarlyStopping có thể dừng trước 50 epoch. Thời gian chạy trên CPU có thể kéo dài nhiều giờ.

In [6]:
%run CNN_LSTM.py --epochs 50 --batch-size 128 --output-dir artifacts_cnn_lstm_rerun

=== DATA PREPARED ===
Train           : 127,638
Val             : 14,183
Test            : 35,456
Obfuscated test : 150,000
Base p99 length : 975

=== TOKENIZER ===
Vocabulary size: 191
Tokenizer was fit on train payloads only.

=== INPUT SHAPES ===
X_train: (127638, 1024)
X_val  : (14183, 1024)
X_test : (35456, 1024)
X_obfu : (150000, 1024)
Class weights: {0: 1.3997543482552146, 1: 0.7778536169175453}


Model: "Hybrid_1D_CNN_LSTM_Web_Attack_Detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_embedding (Embedding)      │ (None, 1024, 64)       │        12,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3 (Conv1D)                │ (None, 1024, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5 (Conv1D)                │ (None, 256, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling1D)           │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_context (LSTM)             │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_classifier (Dense)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attack_probability (Dense)      │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 258,881 (1011.25 KB)

 Trainable params: 258,881 (1011.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
998/998 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.7634 - auc: 0.8564 - loss: 0.4470 - precision: 0.8593 - recall: 0.7510
Epoch 1: val_loss improved from None to 0.06744, saving model to artifacts_cnn_lstm_rerun\best_hybrid_cnn_lstm.keras

Epoch 1: finished saving model to artifacts_cnn_lstm_rerun\best_hybrid_cnn_lstm.keras
998/998 ━━━━━━━━━━━━━━━━━━━━ 6045s 6s/step - accuracy: 0.8553 - auc: 0.9477 - loss: 0.2885 - precision: 0.9207 - recall: 0.8479 - val_accuracy: 0.9729 - val_auc: 0.9974 - val_loss: 0.0674 - val_precision: 0.9652 - val_recall: 0.9937
Epoch 2/50
998/998 ━━━━━━━━━━━━━━━━━━━━ 0s 469ms/step - accuracy: 0.9794 - auc: 0.9967 - loss: 0.0600 - precision: 0.9918 - recall: 0.9760
Epoch 2: val_loss improved from 0.06744 to 0.03849, saving model to artifacts_cnn_lstm_rerun\best_hybrid_cnn_lstm.keras

Epoch 2: finished saving model to artifacts_cnn_lstm_rerun\best_hybrid_cnn_lstm.keras
998/998 ━━━━━━━━━━━━━━━━━━━━ 478s 479ms/step - accuracy: 0.9824 - auc: 0.9975 - l

## 4. Đọc kết quả CNN-LSTM vừa chạy

In [7]:
import json

from pathlib import Path

result_dir = Path('artifacts_cnn_lstm_rerun')
if not (result_dir / 'metadata_and_results.json').exists():
    result_dir = Path('artifacts')

print('Đọc kết quả từ:', result_dir)
with (result_dir / 'metadata_and_results.json').open(encoding='utf-8') as f:
    cnn_lstm_results = json.load(f)

cnn_lstm_results['model'], cnn_lstm_results['evaluation']

({'max_len': 1024,
  'embedding_dim': 64,
  'vocab_size': 191,
  'architecture': 'Embedding -> Conv1D(k3) -> MaxPool(4) -> Conv1D(k5) -> MaxPool(4) -> LSTM(128) -> Dense -> Sigmoid',
  'class_weight': {'0': 1.3997543482552146, '1': 0.7778536169175453},
  'artifacts': {'best_model': 'artifacts_cnn_lstm_rerun\\best_hybrid_cnn_lstm.keras',
   'last_model': 'artifacts_cnn_lstm_rerun\\last_hybrid_cnn_lstm.keras',
   'tokenizer': 'artifacts_cnn_lstm_rerun\\tokenizer.pkl'}},
 {'test': {'accuracy': 0.9942181859205776,
   'auc_roc': 0.9995660250772901,
   'confusion_matrix': [[12630, 35], [170, 22621]],
   'classification_report': {'Normal (0)': {'precision': 0.98671875,
     'recall': 0.9972364784840111,
     'f1-score': 0.9919497349302965,
     'support': 12665.0},
    'Attack (1)': {'precision': 0.9984551553672316,
     'recall': 0.9925409152735729,
     'f1-score': 0.9954892512157018,
     'support': 22791.0},
    'accuracy': 0.9942181859205776,
    'macro avg': {'precision': 0.992586952683

## 5. So sánh CNN-only và CNN-LSTM
Hai model dùng cùng dữ liệu, tokenizer, seed, class weight và threshold 0.5.

In [8]:
import pandas as pd

with open('../cnn_only/artifacts/metadata_and_results.json', encoding='utf-8') as f:
    cnn_results = json.load(f)

cnn_eval = cnn_results['evaluation']
hybrid_eval = cnn_lstm_results['evaluation']
cnn_attack = cnn_eval['normal_test']['classification_report']['Attack (1)']
hybrid_attack = hybrid_eval['test']['classification_report']['Attack (1)']

comparison = pd.DataFrame([
    {
        'Mô hình': 'CNN-only',
        'Tham số': cnn_results['cnn_only_model']['parameter_count'],
        'Test Accuracy': cnn_eval['normal_test']['accuracy'],
        'Test AUC': cnn_eval['normal_test']['auc_roc'],
        'Attack Recall': cnn_attack['recall'],
        'Attack F1': cnn_attack['f1-score'],
        'Obfuscated Recall': cnn_eval['obfuscated_test']['classification_report']['1']['recall'],
        'Obfuscated Missed': cnn_eval['obfuscated_test']['confusion_matrix'][1][0],
    },
    {
        'Mô hình': 'CNN-LSTM rerun',
        'Tham số': None,
        'Test Accuracy': hybrid_eval['test']['accuracy'],
        'Test AUC': hybrid_eval['test']['auc_roc'],
        'Attack Recall': hybrid_attack['recall'],
        'Attack F1': hybrid_attack['f1-score'],
        'Obfuscated Recall': hybrid_eval['obfuscated_test']['classification_report']['1']['recall'],
        'Obfuscated Missed': hybrid_eval['obfuscated_test']['confusion_matrix'][1][0],
    },
])
comparison

,Mô hình,Tham số,Test Accuracy,Test AUC,Attack Recall,Attack F1,Obfuscated Recall,Obfuscated Missed
0,CNN-only,127297.0,0.994105,0.999528,0.992234,0.995400,0.996740,489
1,CNN-LSTM rerun,NaN,0.994218,0.999566,0.992541,0.995489,0.995407,689


## Nguyên tắc kết luận

- Nếu CNN-LSTM tăng rõ Attack Recall hoặc Obfuscated Recall, LSTM có đóng góp thực nghiệm.
- Nếu kết quả gần ngang, ưu tiên CNN-only vì nhẹ hơn.
- Không kết luận chỉ dựa trên accuracy; cần xem False Negative, recall obfuscation, số tham số và thời gian chạy.